# IVF-PQ Quantization — Inverted File Index + Product Quantization

HNSW trades memory for speed. **IVF-PQ** flips the tradeoff:

* **IVF (Inverted File)**: partition the embedding space into `nlist` Voronoi cells (trained with k-means). At query time, probe only the `nprobe` nearest cells.
* **PQ (Product Quantization)**: compress each vector by splitting it into `M` sub-spaces and quantizing each sub-space to `k*` centroids. Reduces memory by 8–32× with modest recall loss.

Together IVF-PQ lets you store 100M vectors in ~20GB RAM and search in milliseconds.
This notebook implements both from scratch using numpy + sklearn.

In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
if not (os.getenv('OPENAI_API_KEY') or os.getenv('ANTHROPIC_API_KEY')):
    os.environ.setdefault('LLM_CACHE_ONLY', '1')
print('LLM_CACHE_ONLY =', os.environ.get('LLM_CACHE_ONLY', '0'))


In [ ]:
import numpy as np
from sklearn.cluster import MiniBatchKMeans
from shared.embedders import cosine_topk, hash_embed
from shared.loaders import load_corpus, load_golden_qa

DOCS = load_corpus()
QA   = [q for q in load_golden_qa() if q.source_ids]
doc_texts = [d.title + '. ' + d.abstract for d in DOCS]
doc_ids   = [d.arxiv_id for d in DOCS]
DIMS = 256
vecs = hash_embed(doc_texts, dims=DIMS, seed=0)
print('vecs:', vecs.shape)

## IVF implementation

In [ ]:
class IVFFlat:
    def __init__(self, vecs, nlist=4):
        nlist = min(nlist, len(vecs))
        norm = vecs / (np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-9)
        self.norm = norm
        self.km = MiniBatchKMeans(n_clusters=nlist, random_state=0, n_init='auto')
        self.assign = self.km.fit_predict(norm)
        self.buckets = {c: np.where(self.assign == c)[0] for c in range(nlist)}
        self.nlist = nlist

    def search(self, qv, k=5, nprobe=2):
        qv = qv / (np.linalg.norm(qv) + 1e-9)
        cent_scores = self.km.cluster_centers_ @ qv
        probed = np.argsort(-cent_scores)[:min(nprobe, self.nlist)]
        cands = np.concatenate([self.buckets.get(c, np.array([], int)) for c in probed])
        if not len(cands):
            return np.array([], int)
        scores = self.norm[cands] @ qv
        return cands[np.argsort(-scores)[:k]]

ivf = IVFFlat(vecs, nlist=3)
print('IVF nlist=3 clusters:', dict(zip(*np.unique(ivf.assign, return_counts=True))))

## Product Quantization (PQ) implementation

Split each 256-d vector into `M` sub-vectors of `dims/M` dimensions. Train `k*` centroids per sub-space. Encode each vector as M byte-sized centroid IDs.

In [ ]:
class ProductQuantizer:
    def __init__(self, vecs, m=8, k_star=256):
        """m sub-spaces, k_star centroids each. vecs shape (N, D)."""
        assert vecs.shape[1] % m == 0, 'dims must be divisible by m'
        self.m = m
        self.k = min(k_star, len(vecs))
        self.d_sub = vecs.shape[1] // m
        self.codebooks: list[Any] = []
        self.codes = np.zeros((len(vecs), m), dtype=np.uint8)
        for i in range(m):
            sub = vecs[:, i * self.d_sub:(i + 1) * self.d_sub].astype(np.float32)
            km = MiniBatchKMeans(n_clusters=self.k, random_state=i, n_init='auto')
            km.fit(sub)
            self.codebooks.append(km.cluster_centers_)
            self.codes[:, i] = km.predict(sub).astype(np.uint8)

    def decode(self, code_row: np.ndarray) -> np.ndarray:
        parts = [self.codebooks[i][code_row[i]] for i in range(self.m)]
        return np.concatenate(parts)

    def approx_distances(self, qv: np.ndarray) -> np.ndarray:
        """Asymmetric Distance Computation (ADC): precompute dist table."""
        dists = np.zeros(len(self.codes), dtype=np.float32)
        for i in range(self.m):
            sub_q = qv[i * self.d_sub:(i + 1) * self.d_sub].astype(np.float32)
            # cosine: dot product after normalization
            table = self.codebooks[i] @ sub_q
            dists += table[self.codes[:, i]]
        return dists

from typing import Any
pq = ProductQuantizer(vecs, m=8, k_star=4)  # k_star=4 keeps it tractable on 10 docs
code_bytes = pq.codes.nbytes
orig_bytes = vecs.nbytes
print(f'Original: {orig_bytes} bytes  PQ codes: {code_bytes} bytes  ratio: {orig_bytes/code_bytes:.1f}x')

## Recall vs nprobe (IVF) and vs PQ approximation

In [ ]:
def recall(retrieve_fn, k=3):
    hits = 0
    for item in QA:
        qv = hash_embed([item.question], dims=DIMS, seed=0)[0]
        got = {doc_ids[i] for i in retrieve_fn(qv, k)}
        if got & set(item.source_ids): hits += 1
    return round(hits / len(QA), 4)

# Flat exact baseline
flat_vecs = vecs / (np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-9)
def flat_fn(qv, k): return cosine_topk(qv, vecs, k)[0]
print(f'Flat exact recall@3: {recall(flat_fn)}')

# IVF nprobe sweep
print('\nnprobe sweep (IVF nlist=3):')
for np_val in [1, 2, 3]:
    fn = lambda qv, k, np_=np_val: ivf.search(qv, k=k, nprobe=np_)
    print(f'  nprobe={np_val}  recall@3={recall(fn)}')

# PQ approximate search
def pq_fn(qv, k):
    approx = pq.approx_distances(qv)
    return np.argsort(-approx)[:k]
print(f'\nPQ approx recall@3: {recall(pq_fn)}')

## Memory vs recall tradeoff (conceptual)

| Config | Memory | recall@3 |
|---|---|---|
| Flat (float32) | 100% | exact |
| IVF nprobe=1 | 100% store, less compute | ≈ flat |
| PQ m=8 k=256 | ~3% | small loss |
| PQ m=4 k=256 | ~1.5% | larger loss |

At scale (100M docs) the tradeoff becomes compelling: 100M × 256 × 4 bytes = 100GB → PQ reduces to ~3GB.

## Production notes

* FAISS `IndexIVFPQ` uses the same algorithm but with optimized C++ BLAS.
* `nlist = 4√N` is a common rule-of-thumb for choosing the number of centroids.
* OPQ (Optimized PQ) rotates the vector space before quantization to minimize quantization error.
* For 1B-scale: Faiss + GPU (IndexIVFPQ on GPU reduces query time by 100×).